# rapid-agent: production governance for Gemini

This notebook demonstrates the four governance layers in rapid-agent:
1. **Structured output enforcement** — typed `Brief` objects, auto-repair on bad JSON
2. **Budget cap** — per-run USD ceiling, pre-flight projection
3. **Egress allowlist** — blocks unauthorized hostnames before the socket opens
4. **Arize Phoenix trace** — every fetch and model call as an OTLP span

No API key required — the demo uses `StubClient` (canned responses).

In [ ]:
# Install rapid-agent with Arize Phoenix observability
!pip install -q 'git+https://github.com/MukundaKatta/rapid-agent.git#egg=rapid-agent[phoenix]'

In [ ]:
import json
from rapid_agent import (
    RapidAgent, StubClient, Brief,
    BudgetCap, EgressAllowlist, Trace,
)

# --- governance config ---
budget  = BudgetCap(usd_limit=0.10)          # hard $0.10 ceiling for this run
egress  = EgressAllowlist(['example.com', 'wikipedia.org'])  # only these hosts
trace   = Trace()

agent = RapidAgent(
    client=StubClient(),   # swap for GeminiClient() when you have a key
    budget=budget,
    allowlist=egress,
    trace=trace,
)

urls = [
    'https://example.com/ai-safety',       # allowed
    'https://wikipedia.org/wiki/Gemini',   # allowed
    'https://internal-admin.corp/secret',  # BLOCKED by egress allowlist
]

print('Running rapid-agent demo...')
brief: Brief = agent.run('AI safety in production', urls=urls)

print('\n=== Brief ===')
for item in brief.items:
    print(f'  [{item.title}] {item.summary}')

print('\n=== Trace ===')
print(json.dumps(trace.to_dict(), indent=2))

In [ ]:
# Optional: export trace to Arize Phoenix
# Start Phoenix first:  python -m phoenix.server.main &
# Then run this cell:
try:
    from rapid_agent.phoenix_export import export_trace_to_phoenix
    ok = export_trace_to_phoenix(trace, run_name='colab-demo')
    print('Phoenix export:', 'OK' if ok else 'skipped (arize-phoenix not installed)')
except Exception as e:
    print(f'Phoenix not running locally: {e}')

## Swap in a real Gemini key

```python
import os
from rapid_agent import GeminiClient

client = GeminiClient(api_key=os.environ['GEMINI_API_KEY'])
agent  = RapidAgent(client=client, budget=budget, allowlist=egress, trace=trace)
brief  = agent.run('AI safety in production', urls=urls)
```

## Deploy to Cloud Run (Vertex AI)

See [DEPLOY.md](https://github.com/MukundaKatta/rapid-agent/blob/main/DEPLOY.md) for
a step-by-step guide: swap `GeminiClient` for `VertexClient`, add a Dockerfile, and
`gcloud run deploy`.